# Bronze Ingestion
This notebook implements Bronze-layer ingestion from ADLS Gen2 using Databricks Auto Loader.
Auto Loader is used to support incremental file ingestion and checkpoint-based processing, which makes the notebook suitable for repeated execution as a Databricks Job.

In [0]:
from pyspark.sql import functions as F

storage_account = "dlsua5816bd"
container = "victoriya44250"

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

climatic_raw_path = base_path + "/raw_data/yield_climate/"
climatic_checkpoint_path = base_path + "/checkpoints/yield_climate/"
climatic_schema_path = base_path + "/schemas/yield_climate/"

target_table = "dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.yield_and_climate"

### Read source data

In [0]:
df_climatic = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", climatic_schema_path)
    .load(climatic_raw_path)
)

### Add ingestion metadata

In [0]:
df_climatic_bronze = (
    df_climatic
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

### Write to the Bronze layer

In [0]:
query = (
    df_climatic_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", climatic_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

### Verify the result

In [0]:
display(
    spark.table("dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.yield_and_climate")
)

## Repeating the same loop for another table

In [0]:
environment_raw_path = base_path + "/raw_data/environment_factors/"
environment_checkpoint_path = base_path + "/checkpoints/environment_factors/"
environment_schema_path = base_path + "/schemas/environment_factors/"

environment_target_table = "dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.environment"

%md
### Read source data

In [0]:
df_environment = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", environment_schema_path)
    .load(environment_raw_path)
)

### Add ingestion metadata

In [0]:
df_environment_bronze = (
    df_environment
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

### Write to the Bronze layer

In [0]:
environment_query = (
    df_environment_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", environment_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(environment_target_table)
)

environment_query.awaitTermination()

### Verify the result

In [0]:
display(
    spark.table("dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.environment")
)

### Checking:

In [0]:
print(
    "climatic:",
    spark.table("dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.yield_and_climate").count()
)

print(
    "environment:",
    spark.table("dbr_dev_ua5816bd.viktoriia_kalenichenko_bronze.environment").count()
)